# Fixed Force Length Measurement Analysis

Analysis of length measurements from fixed force experiments with cyclic voltage patterns.

**Expected behavior:**
- ~30 second initial offset before voltage cycles begin
- Alternating 300-second cycles: -0.8V → +0.6V → repeat
- Linear relationship: `length = -0.04130 × voltage` (accounting for 10x gain)
- Channel 1 = length measurement, Channel 2 = force/control voltage

**Analysis includes:**
- Automatic detection of cycle start offset
- Noise reduction with median filtering
- Validation of expected linear relationship
- Cycle-by-cycle analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, stats
from scipy.ndimage import median_filter
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = [12, 8]

In [ ]:
# Load the data
df = pd.read_csv('20260430_fixedforce_length_measurement.csv')
print(f"Data shape: {df.shape}")
print(f"Time range: {df['elapsed_s'].min():.2f} - {df['elapsed_s'].max():.2f} seconds")
print(f"Duration: {(df['elapsed_s'].max() - df['elapsed_s'].min())/60:.1f} minutes")

# Basic statistics
print("\nChannel Statistics:")
print(f"Channel 1 (Length): {df['ch1_MEAN_V'].min():.3f} to {df['ch1_MEAN_V'].max():.3f}V")
print(f"Channel 2 (Control): {df['ch2_MEAN_V'].min():.3f} to {df['ch2_MEAN_V'].max():.3f}V")

print("\nFirst few rows:")
print(df.head(10))

In [ ]:
# Plot raw data overview
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# Channel 1 (Length)
ax1.plot(df['elapsed_s'], df['ch1_MEAN_V'], alpha=0.7, linewidth=1)
ax1.set_ylabel('Length Voltage (V)', fontsize=12)
ax1.set_title('Channel 1: Length Measurement Over Time', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Channel 2 (Control)
ax2.plot(df['elapsed_s'], df['ch2_MEAN_V'], alpha=0.7, linewidth=1, color='orange')
ax2.set_ylabel('Control Voltage (V)', fontsize=12)
ax2.set_xlabel('Time (seconds)', fontsize=12)
ax2.set_title('Channel 2: Control/Force Voltage', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quick stats
print(f"Sampling rate: ~{len(df) / (df['elapsed_s'].max() - df['elapsed_s'].min()):.1f} Hz")
print(f"Data points: {len(df)}")

In [ ]:
# Detect voltage level changes to find cycle boundaries
# We'll use channel 2 as the control signal to identify cycles

# Apply median filter to control channel for better edge detection
control_filtered = median_filter(df['ch2_MEAN_V'], size=21)  # ~21 point median filter

# Calculate derivative to find sudden changes
control_diff = np.diff(control_filtered)
control_diff_abs = np.abs(control_diff)

# Find significant voltage changes (threshold based on data)
change_threshold = np.std(control_diff_abs) * 3  # 3 sigma threshold
significant_changes = np.where(control_diff_abs > change_threshold)[0]

print(f"Detected {len(significant_changes)} significant voltage changes")
print(f"Change threshold: {change_threshold:.4f}V")

if len(significant_changes) > 0:
    print(f"First significant change at: {df['elapsed_s'].iloc[significant_changes[0]]:.1f} seconds")
    print(f"This is likely the start of the voltage cycles (after initial offset)")
    
    cycle_start_time = df['elapsed_s'].iloc[significant_changes[0]]
    initial_offset = cycle_start_time
    
    print(f"\nEstimated initial offset: {initial_offset:.1f} seconds")
else:
    print("No significant voltage changes detected. Using manual offset estimate.")
    initial_offset = 30  # fallback to expected 30 seconds
    cycle_start_time = initial_offset

In [ ]:
# Apply median filtering to reduce noise in length measurements
# Keep original data for comparison
filter_size = 15  # Adjust as needed for noise reduction vs. responsiveness

length_filtered = median_filter(df['ch1_MEAN_V'], size=filter_size)
control_smooth = median_filter(df['ch2_MEAN_V'], size=filter_size)

print(f"Applied median filter with window size: {filter_size} points")
print(f"Noise reduction (std): {df['ch1_MEAN_V'].std():.4f} → {length_filtered.std():.4f}V")

# Add filtered data to dataframe for convenience
df['ch1_filtered'] = length_filtered
df['ch2_filtered'] = control_smooth

In [ ]:
# Create comprehensive plot showing raw vs filtered data
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 12))

# Channel 1: Length with filtering
ax1.plot(df['elapsed_s'], df['ch1_MEAN_V'], alpha=0.3, linewidth=1, 
         color='lightblue', label='Raw Data')
ax1.plot(df['elapsed_s'], df['ch1_filtered'], linewidth=2, 
         color='darkblue', label='Median Filtered')
ax1.axvline(x=cycle_start_time, color='red', linestyle='--', alpha=0.8,
           label=f'Cycle Start (~{cycle_start_time:.1f}s)')
ax1.set_ylabel('Length Voltage (V)', fontsize=12)
ax1.set_title('Channel 1: Length Measurement (Raw vs. Filtered)', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Channel 2: Control signal
ax2.plot(df['elapsed_s'], df['ch2_MEAN_V'], alpha=0.3, linewidth=1, 
         color='lightcoral', label='Raw Data')
ax2.plot(df['elapsed_s'], df['ch2_filtered'], linewidth=2, 
         color='darkred', label='Median Filtered')
ax2.axvline(x=cycle_start_time, color='red', linestyle='--', alpha=0.8)
ax2.set_ylabel('Control Voltage (V)', fontsize=12)
ax2.set_title('Channel 2: Control Signal (Raw vs. Filtered)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Phase plot: Length vs Control (after cycle start)
cycle_mask = df['elapsed_s'] >= cycle_start_time
ax3.scatter(df[cycle_mask]['ch2_filtered'], df[cycle_mask]['ch1_filtered'], 
           alpha=0.6, s=20, c=df[cycle_mask]['elapsed_s'], cmap='viridis')
ax3.set_xlabel('Control Voltage (V)', fontsize=12)
ax3.set_ylabel('Length Voltage (V)', fontsize=12)
ax3.set_title('Phase Plot: Length vs. Control (Colored by Time)', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# Add colorbar for time
cbar = plt.colorbar(ax3.collections[0], ax=ax3)
cbar.set_label('Time (seconds)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze the linear relationship: length = -0.04130 * voltage
# Focus on data after the initial offset
cycle_data = df[df['elapsed_s'] >= cycle_start_time].copy()

# Use filtered data for analysis
X = cycle_data['ch2_filtered'].values.reshape(-1, 1)
y = cycle_data['ch1_filtered'].values

# Fit linear model
model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)
r2 = r2_score(y, y_pred)

# Extract coefficients
measured_slope = model.coef_[0]
measured_intercept = model.intercept_
expected_slope = -0.04130

# Calculate statistics
correlation, p_value = stats.pearsonr(cycle_data['ch2_filtered'], cycle_data['ch1_filtered'])
rmse = np.sqrt(np.mean((y - y_pred)**2))
mae = np.mean(np.abs(y - y_pred))

print("Linear Relationship Analysis:")
print("=" * 50)
print(f"Expected: length = {expected_slope:.5f} × voltage")
print(f"Measured: length = {measured_slope:.5f} × voltage + {measured_intercept:.5f}")
print(f"Slope difference: {abs(measured_slope - expected_slope):.5f} ({abs(measured_slope - expected_slope)/abs(expected_slope)*100:.1f}% error)")
print(f"R² = {r2:.6f}")
print(f"Correlation = {correlation:.6f} (p = {p_value:.2e})")
print(f"RMSE = {rmse:.6f}V")
print(f"MAE = {mae:.6f}V")
print(f"Data points analyzed: {len(cycle_data)}")

In [ ]:
# Create detailed linear relationship plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Scatter plot with fitted line
scatter = ax1.scatter(cycle_data['ch2_filtered'], cycle_data['ch1_filtered'], 
                     alpha=0.5, s=20, c=cycle_data['elapsed_s'], cmap='plasma')

# Plot both expected and measured lines
voltage_range = np.linspace(cycle_data['ch2_filtered'].min(), 
                           cycle_data['ch2_filtered'].max(), 100)
expected_line = expected_slope * voltage_range
measured_line = measured_slope * voltage_range + measured_intercept

ax1.plot(voltage_range, expected_line, 'r--', linewidth=3, 
         label=f'Expected: {expected_slope:.5f}x', alpha=0.8)
ax1.plot(voltage_range, measured_line, 'g-', linewidth=3, 
         label=f'Measured: {measured_slope:.5f}x + {measured_intercept:.3f}', alpha=0.8)

ax1.set_xlabel('Control Voltage (V)', fontsize=12)
ax1.set_ylabel('Length Voltage (V)', fontsize=12)
ax1.set_title(f'Linear Relationship (R² = {r2:.4f})', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Add colorbar
cbar1 = plt.colorbar(scatter, ax=ax1)
cbar1.set_label('Time (seconds)', fontsize=12)

# Residuals plot
residuals = y - y_pred
ax2.scatter(y_pred, residuals, alpha=0.6, s=20)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.8)
ax2.set_xlabel('Predicted Length Voltage (V)', fontsize=12)
ax2.set_ylabel('Residuals (V)', fontsize=12)
ax2.set_title('Residuals vs. Predicted Values', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Add text box with statistics
textstr = f'Slope Error: {abs(measured_slope - expected_slope)/abs(expected_slope)*100:.1f}%\nRMSE: {rmse:.6f}V\nMAE: {mae:.6f}V'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax2.text(0.05, 0.95, textstr, transform=ax2.transAxes, fontsize=11,
         verticalalignment='top', bbox=props)

plt.tight_layout()
plt.show()

In [ ]:
# Create expected input voltage pattern
# Pattern: -0.8V for 300s, then +0.6V for 300s, repeating
print("Creating Expected Input Voltage Pattern:")
print("=" * 50)

# Create time vector starting from cycle start
cycle_time_full = df['elapsed_s'] - cycle_start_time
expected_input = np.zeros_like(df['elapsed_s'])

# Fill in expected pattern for times after cycle start
cycle_mask = df['elapsed_s'] >= cycle_start_time
cycle_times = cycle_time_full[cycle_mask]

for i, t in enumerate(cycle_times):
    # Determine which 600-second cycle we're in
    cycle_number = int(t // 600)  # Each full cycle is 600s (300s + 300s)
    time_in_cycle = t % 600       # Time within current 600s cycle
    
    if time_in_cycle < 300:
        expected_input[cycle_mask.index[i]] = -0.8  # First 300s: -0.8V
    else:
        expected_input[cycle_mask.index[i]] = 0.6   # Next 300s: +0.6V

# Before cycle start, set to baseline (assume 0V or first value)
pre_cycle_mask = df['elapsed_s'] < cycle_start_time
expected_input[pre_cycle_mask] = 0.0  # Baseline before cycles

# Add to dataframe
df['expected_input'] = expected_input

print(f"Expected input pattern created for {cycle_times.iloc[-1]/60:.1f} minutes of cycling")
print(f"Number of complete 600s cycles: {int(cycle_times.iloc[-1] // 600)}")
print(f"Expected voltage levels: {np.unique(expected_input[cycle_mask])}")

# Quick validation
actual_control = df[cycle_mask]['ch2_filtered']
expected_control = expected_input[cycle_mask]
agreement = np.mean(np.abs(actual_control - expected_control) < 0.2)  # Within 0.2V
print(f"Actual vs. expected agreement: {agreement*100:.1f}% (within 0.2V tolerance)")

In [ ]:
# Linear drift correction analysis
# Sample length data every 600 seconds and fit linear drift trend
print("Linear Drift Correction Analysis:")
print("=" * 50)

# Get cycle data for drift analysis
cycle_data_drift = df[df['elapsed_s'] >= cycle_start_time].copy()
cycle_times_drift = cycle_data_drift['elapsed_s'] - cycle_start_time

# Sample points every 600 seconds (complete cycles)
sampling_interval = 600  # seconds
max_cycle_time = cycle_times_drift.iloc[-1]
sample_times = np.arange(0, max_cycle_time + sampling_interval, sampling_interval)

# Find actual data points closest to sample times
sample_indices = []
sample_values = []
sample_timestamps = []

for target_time in sample_times:
    if target_time <= max_cycle_time:
        # Find closest data point to target time
        time_diffs = np.abs(cycle_times_drift - target_time)
        closest_idx = time_diffs.idxmin()
        closest_time = cycle_times_drift.loc[closest_idx]
        closest_value = cycle_data_drift.loc[closest_idx, 'ch1_filtered']
        
        sample_indices.append(closest_idx)
        sample_values.append(closest_value)
        sample_timestamps.append(closest_time)
        
        print(f"Sample at t={closest_time:.1f}s: {closest_value:.4f}V")

sample_values = np.array(sample_values)
sample_timestamps = np.array(sample_timestamps)

print(f"\nCollected {len(sample_values)} drift calibration points")

# Fit linear trend to sample points
if len(sample_values) >= 2:
    drift_model = LinearRegression()
    drift_model.fit(sample_timestamps.reshape(-1, 1), sample_values)
    
    drift_slope = drift_model.coef_[0]
    drift_intercept = drift_model.intercept_
    drift_r2 = drift_model.score(sample_timestamps.reshape(-1, 1), sample_values)
    
    print(f"\nDrift Analysis Results:")
    print(f"Drift slope: {drift_slope:.6f} V/s ({drift_slope*3600:.4f} V/hour)")
    print(f"Drift intercept: {drift_intercept:.6f} V")
    print(f"Drift fit R²: {drift_r2:.4f}")
    
    # Calculate drift correction for entire dataset
    drift_correction = drift_slope * cycle_times_drift + drift_intercept
    
    # Apply drift correction to filtered length data
    corrected_length = cycle_data_drift['ch1_filtered'] - drift_correction + drift_intercept
    cycle_data_drift['ch1_drift_corrected'] = corrected_length
    
    # Calculate drift statistics
    total_drift = drift_slope * max_cycle_time
    print(f"Total drift over {max_cycle_time/60:.1f} minutes: {total_drift:.6f} V")
    
    # Assess drift significance
    length_range = cycle_data_drift['ch1_filtered'].max() - cycle_data_drift['ch1_filtered'].min()
    drift_significance = abs(total_drift) / length_range * 100
    print(f"Drift as % of signal range: {drift_significance:.1f}%")
    
    if drift_significance > 5:
        print("⚠️  Significant drift detected (>5% of signal range)")
    else:
        print("✅ Drift is relatively small (<5% of signal range)")
        
else:
    print("❌ Not enough sample points for drift analysis")
    corrected_length = cycle_data_drift['ch1_filtered']  # No correction
    drift_slope = 0
    drift_intercept = cycle_data_drift['ch1_filtered'].iloc[0]

In [ ]:
# Plot expected input pattern and drift-corrected results
fig = plt.figure(figsize=(16, 14))

# Create subplot grid
gs = fig.add_gridspec(4, 2, height_ratios=[1, 1, 1, 1], hspace=0.3, wspace=0.3)

# 1. Expected vs Actual Input Voltage (full timeline)
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(df['elapsed_s'], df['expected_input'], linewidth=3, color='green', 
         label='Expected Input Pattern', alpha=0.8)
ax1.plot(df['elapsed_s'], df['ch2_filtered'], linewidth=2, color='orange', 
         label='Actual Control Voltage', alpha=0.8)
ax1.axvline(x=cycle_start_time, color='red', linestyle='--', alpha=0.7, 
           label=f'Cycle Start ({cycle_start_time:.1f}s)')
ax1.set_ylabel('Voltage (V)', fontsize=12)
ax1.set_title('Expected vs. Actual Input Voltage Pattern', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Original Length Data with Drift Line
ax2 = fig.add_subplot(gs[1, :])
cycle_times_plot = cycle_data_drift['elapsed_s']
ax2.plot(cycle_times_plot, cycle_data_drift['ch1_MEAN_V'], alpha=0.3, 
         color='lightblue', linewidth=1, label='Raw Length Data')
ax2.plot(cycle_times_plot, cycle_data_drift['ch1_filtered'], 
         color='blue', linewidth=2, label='Filtered Length Data')

# Plot drift line
if len(sample_values) >= 2:
    drift_line_full = drift_slope * cycle_times_drift + drift_intercept
    ax2.plot(cycle_times_plot, drift_line_full, color='red', linewidth=3, 
             linestyle='--', label=f'Linear Drift (slope={drift_slope:.6f}V/s)', alpha=0.8)
    
    # Mark sample points
    sample_cycle_times = cycle_data_drift.iloc[[i for i in range(len(cycle_data_drift)) 
                                               if cycle_data_drift.index[i] in sample_indices]]['elapsed_s']
    ax2.scatter(sample_cycle_times, sample_values, color='red', s=80, 
               zorder=5, label=f'Drift Sample Points (every 600s)')

ax2.set_ylabel('Length Voltage (V)', fontsize=12)
ax2.set_title('Original Length Data with Linear Drift Trend', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Drift-Corrected Length Data
ax3 = fig.add_subplot(gs[2, :])
ax3.plot(cycle_times_plot, cycle_data_drift['ch1_filtered'], alpha=0.4, 
         color='lightblue', linewidth=2, label='Original (Filtered)')
if len(sample_values) >= 2:
    ax3.plot(cycle_times_plot, cycle_data_drift['ch1_drift_corrected'], 
             color='darkgreen', linewidth=2, label='Drift Corrected')
ax3.set_ylabel('Length Voltage (V)', fontsize=12)
ax3.set_title('Drift-Corrected Length Measurements', fontsize=14, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Before vs After Drift Correction Comparison
ax4a = fig.add_subplot(gs[3, 0])
ax4a.hist(cycle_data_drift['ch1_filtered'], bins=50, alpha=0.6, 
          color='blue', label='Original', density=True)
if len(sample_values) >= 2:
    ax4a.hist(cycle_data_drift['ch1_drift_corrected'], bins=50, alpha=0.6, 
              color='green', label='Drift Corrected', density=True)
ax4a.set_xlabel('Length Voltage (V)', fontsize=12)
ax4a.set_ylabel('Density', fontsize=12)
ax4a.set_title('Distribution Comparison', fontsize=12, fontweight='bold')
ax4a.legend()
ax4a.grid(True, alpha=0.3)

# 5. Linear Relationship with Drift Correction
ax4b = fig.add_subplot(gs[3, 1])
if len(sample_values) >= 2:
    # Use drift-corrected data for relationship analysis
    corrected_cycle_data = cycle_data_drift.dropna(subset=['ch1_drift_corrected'])
    ax4b.scatter(corrected_cycle_data['ch2_filtered'], corrected_cycle_data['ch1_drift_corrected'], 
                alpha=0.5, s=20, c=corrected_cycle_data['elapsed_s'], cmap='plasma', label='Drift Corrected')
    
    # Fit new relationship with corrected data
    X_corrected = corrected_cycle_data['ch2_filtered'].values.reshape(-1, 1)
    y_corrected = corrected_cycle_data['ch1_drift_corrected'].values
    model_corrected = LinearRegression()
    model_corrected.fit(X_corrected, y_corrected)
    
    corrected_slope = model_corrected.coef_[0]
    corrected_intercept = model_corrected.intercept_
    corrected_r2 = model_corrected.score(X_corrected, y_corrected)
    
    # Plot corrected fit line
    voltage_range = np.linspace(corrected_cycle_data['ch2_filtered'].min(), 
                               corrected_cycle_data['ch2_filtered'].max(), 100)
    corrected_line = corrected_slope * voltage_range + corrected_intercept
    expected_line = expected_slope * voltage_range
    
    ax4b.plot(voltage_range, expected_line, 'r--', linewidth=3, 
             label=f'Expected: {expected_slope:.5f}x', alpha=0.8)
    ax4b.plot(voltage_range, corrected_line, 'g-', linewidth=3, 
             label=f'Corrected: {corrected_slope:.5f}x + {corrected_intercept:.3f}\\nR² = {corrected_r2:.4f}', alpha=0.8)

ax4b.set_xlabel('Control Voltage (V)', fontsize=12)
ax4b.set_ylabel('Length Voltage (V)', fontsize=12)
ax4b.set_title('Linear Relationship (Drift Corrected)', fontsize=12, fontweight='bold')
ax4b.legend(fontsize=10)
ax4b.grid(True, alpha=0.3)

plt.suptitle('Complete Analysis: Input Pattern, Drift Correction & Linear Relationship', 
             fontsize=16, fontweight='bold', y=0.98)
plt.show()

# Print comparison statistics
if len(sample_values) >= 2:
    print(\"\\nDRIFT CORRECTION IMPACT:\")\n    print(\"=\" * 50)\n    original_std = cycle_data_drift['ch1_filtered'].std()\n    corrected_std = cycle_data_drift['ch1_drift_corrected'].std()\n    std_reduction = (original_std - corrected_std) / original_std * 100\n    \n    print(f\"Original std: {original_std:.6f}V\")\n    print(f\"Corrected std: {corrected_std:.6f}V\")\n    print(f\"Std reduction: {std_reduction:.1f}%\")\n    \n    print(f\"\\nLINEAR FIT IMPROVEMENT:\")\n    print(f\"Original R²: {r2:.6f}\")\n    print(f\"Corrected R²: {corrected_r2:.6f}\")\n    print(f\"R² improvement: {corrected_r2 - r2:.6f}\")\n    \n    print(f\"\\nSLOPE ACCURACY:\")\n    original_error = abs(measured_slope - expected_slope) / abs(expected_slope) * 100\n    corrected_error = abs(corrected_slope - expected_slope) / abs(expected_slope) * 100\n    print(f\"Original slope error: {original_error:.2f}%\")\n    print(f\"Corrected slope error: {corrected_error:.2f}%\")\n    print(f\"Error improvement: {original_error - corrected_error:.2f} percentage points\")

In [ ]:
# Analyze cycling behavior - detect individual cycles
# Look for transitions in the control voltage

# Focus on cycle data
cycle_time = cycle_data['elapsed_s'] - cycle_start_time  # Time since cycles started
control_signal = cycle_data['ch2_filtered']
length_signal = cycle_data['ch1_filtered']

# Detect voltage level changes
control_diff = np.abs(np.diff(control_signal))
change_threshold = np.percentile(control_diff, 99)  # Top 1% of changes
transition_indices = np.where(control_diff > change_threshold)[0]

print(f"Cycle Analysis:")
print(f"Cycle duration: {cycle_time.iloc[-1]:.1f} seconds")
print(f"Detected {len(transition_indices)} major transitions")

# Estimate cycle periods
if len(transition_indices) > 1:
    transition_times = cycle_time.iloc[transition_indices]
    cycle_periods = np.diff(transition_times)
    
    print(f"Transition times: {[f'{t:.1f}s' for t in transition_times[:10]]}...")  # First 10
    if len(cycle_periods) > 0:
        print(f"Cycle periods: {cycle_periods[:5]:.1f} seconds (first 5)")
        print(f"Average cycle period: {np.mean(cycle_periods):.1f} ± {np.std(cycle_periods):.1f} seconds")
        print(f"Expected: ~300 seconds per phase")
        
        # Check if we're close to expected 300s cycles
        avg_period = np.mean(cycle_periods)
        if abs(avg_period - 300) < 50:  # Within 50 seconds
            print("✅ Cycle periods match expected ~300 second phases")
        else:
            print("⚠️  Cycle periods don't match expected 300 seconds")

In [ ]:
# Plot time series with cycle markers
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10))

# Length signal over time with cycle markers
ax1.plot(cycle_time, length_signal, linewidth=2, color='blue', label='Filtered Length')
ax1.plot(cycle_time, cycle_data['ch1_MEAN_V'] - cycle_start_time, alpha=0.3, 
         color='lightblue', label='Raw Length')

# Mark transitions
if len(transition_indices) > 0:
    for i, idx in enumerate(transition_indices[:10]):  # First 10 transitions
        ax1.axvline(x=cycle_time.iloc[idx], color='red', linestyle='--', 
                   alpha=0.7, linewidth=1)
        if i == 0:  # Add label only for first line
            ax1.axvline(x=cycle_time.iloc[idx], color='red', linestyle='--', 
                       alpha=0.7, linewidth=1, label='Detected Transitions')

ax1.set_ylabel('Length Voltage (V)', fontsize=12)
ax1.set_title('Length Response During Voltage Cycles', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Control signal with cycle markers
ax2.plot(cycle_time, control_signal, linewidth=2, color='orange', label='Filtered Control')
ax2.plot(cycle_time, cycle_data['ch2_MEAN_V'] - cycle_start_time, alpha=0.3, 
         color='moccasin', label='Raw Control')

# Mark transitions
if len(transition_indices) > 0:
    for i, idx in enumerate(transition_indices[:10]):
        ax2.axvline(x=cycle_time.iloc[idx], color='red', linestyle='--', 
                   alpha=0.7, linewidth=1)
        if i == 0:
            ax2.axvline(x=cycle_time.iloc[idx], color='red', linestyle='--', 
                       alpha=0.7, linewidth=1, label='Detected Transitions')

# Add expected voltage levels
ax2.axhline(y=-0.8, color='green', linestyle=':', alpha=0.8, label='Expected: -0.8V')
ax2.axhline(y=0.6, color='green', linestyle=':', alpha=0.8, label='Expected: +0.6V')

ax2.set_xlabel('Time Since Cycle Start (seconds)', fontsize=12)
ax2.set_ylabel('Control Voltage (V)', fontsize=12)
ax2.set_title('Control Voltage Cycling Pattern', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics and validation
print("EXPERIMENT SUMMARY")
print("=" * 60)
print(f"📊 Total experiment duration: {df['elapsed_s'].max()/60:.1f} minutes")
print(f"⏱️  Initial offset before cycles: {initial_offset:.1f} seconds")
print(f"🔄 Active cycling duration: {cycle_time.iloc[-1]/60:.1f} minutes")
print(f"📈 Data points: {len(df)} total, {len(cycle_data)} in cycles")

print("\nLINEAR RELATIONSHIP VALIDATION")
print("=" * 60)
print(f"🎯 Expected slope: {expected_slope:.5f}")
print(f"📏 Measured slope: {measured_slope:.5f}")
print(f"📊 Slope accuracy: {100 - abs(measured_slope - expected_slope)/abs(expected_slope)*100:.1f}%")
print(f"📈 R² fit quality: {r2:.4f}")
print(f"📉 RMSE: {rmse:.6f}V")

# Voltage level analysis
control_levels = cycle_data['ch2_filtered']
unique_levels = np.unique(np.round(control_levels, 1))
print(f"\nCONTROL VOLTAGE LEVELS")
print("=" * 60)
print(f"Detected voltage levels: {unique_levels}")
print(f"Expected levels: [-0.8, +0.6]")

# Check if we hit expected levels
min_voltage = control_levels.min()
max_voltage = control_levels.max()
print(f"Actual range: {min_voltage:.2f}V to {max_voltage:.2f}V")

if abs(min_voltage - (-0.8)) < 0.1:
    print("✅ Low voltage level matches expected -0.8V")
else:
    print(f"⚠️  Low voltage ({min_voltage:.2f}V) differs from expected -0.8V")

if abs(max_voltage - 0.6) < 0.1:
    print("✅ High voltage level matches expected +0.6V")
else:
    print(f"⚠️  High voltage ({max_voltage:.2f}V) differs from expected +0.6V")

print("\nNOISE REDUCTION")
print("=" * 60)
original_noise = df['ch1_MEAN_V'].std()
filtered_noise = df['ch1_filtered'].std()
noise_reduction = (original_noise - filtered_noise) / original_noise * 100
print(f"Original noise (std): {original_noise:.6f}V")
print(f"Filtered noise (std): {filtered_noise:.6f}V")
print(f"Noise reduction: {noise_reduction:.1f}%")

In [ ]:
# Summary statistics and validation (updated with drift correction)
print("EXPERIMENT SUMMARY (WITH DRIFT CORRECTION)")
print("=" * 60)
print(f"📊 Total experiment duration: {df['elapsed_s'].max()/60:.1f} minutes")
print(f"⏱️  Initial offset before cycles: {initial_offset:.1f} seconds")
print(f"🔄 Active cycling duration: {cycle_time.iloc[-1]/60:.1f} minutes")
print(f"📈 Data points: {len(df)} total, {len(cycle_data)} in cycles")

print("\\nINPUT PATTERN VALIDATION")
print("=" * 60)
print(f"🎯 Expected pattern: -0.8V (300s) → +0.6V (300s) → repeat")
actual_agreement = np.mean(np.abs(df[cycle_mask]['ch2_filtered'] - df[cycle_mask]['expected_input']) < 0.2)
print(f"📊 Pattern agreement: {actual_agreement*100:.1f}% (within 0.2V tolerance)")

if len(sample_values) >= 2:
    print("\\nDRIFT ANALYSIS")
    print("=" * 60)
    print(f"📏 Drift rate: {drift_slope:.6f} V/s ({drift_slope*3600:.4f} V/hour)")
    print(f"📈 Total drift: {total_drift:.6f}V over {max_cycle_time/60:.1f} minutes")
    print(f"📊 Drift significance: {drift_significance:.1f}% of signal range")
    if drift_significance > 5:
        print("⚠️  Significant drift detected - correction applied")
    else:
        print("✅ Minor drift - correction still applied for precision")

print("\\nLINEAR RELATIONSHIP VALIDATION")
print("=" * 60)
print(f"🎯 Expected slope: {expected_slope:.5f}")
print(f"📏 Original measured: {measured_slope:.5f} (R² = {r2:.4f})")

if len(sample_values) >= 2:
    print(f"📈 Drift-corrected: {corrected_slope:.5f} (R² = {corrected_r2:.4f})")
    print(f"🎯 Original accuracy: {100 - abs(measured_slope - expected_slope)/abs(expected_slope)*100:.1f}%")
    print(f"🎯 Corrected accuracy: {100 - abs(corrected_slope - expected_slope)/abs(expected_slope)*100:.1f}%")
    print(f"📊 R² improvement: {corrected_r2 - r2:.6f}")
else:
    print(f"📊 Slope accuracy: {100 - abs(measured_slope - expected_slope)/abs(expected_slope)*100:.1f}%")

print("\\nNOISE REDUCTION")
print("=" * 60)
original_noise = df['ch1_MEAN_V'].std()
filtered_noise = df['ch1_filtered'].std()
noise_reduction = (original_noise - filtered_noise) / original_noise * 100
print(f"📉 Median filter: {noise_reduction:.1f}% noise reduction")

if len(sample_values) >= 2:
    drift_noise_reduction = (cycle_data_drift['ch1_filtered'].std() - 
                             cycle_data_drift['ch1_drift_corrected'].std()) / cycle_data_drift['ch1_filtered'].std() * 100
    print(f"📉 Drift correction: {drift_noise_reduction:.1f}% additional std reduction")

# Voltage level analysis
control_levels = cycle_data['ch2_filtered']
print(f"\\nCONTROL VOLTAGE LEVELS")
print("=" * 60)
min_voltage = control_levels.min()
max_voltage = control_levels.max()
print(f"🔍 Actual range: {min_voltage:.2f}V to {max_voltage:.2f}V")
print(f"🎯 Expected range: -0.8V to +0.6V")

if abs(min_voltage - (-0.8)) < 0.1:
    print("✅ Low voltage level matches expected -0.8V")
else:
    print(f"⚠️  Low voltage ({min_voltage:.2f}V) differs from expected -0.8V")

if abs(max_voltage - 0.6) < 0.1:
    print("✅ High voltage level matches expected +0.6V")
else:
    print(f"⚠️  High voltage ({max_voltage:.2f}V) differs from expected +0.6V")